# CS-31 Environment Check

Run every cell in order (**Run All** at the top). The last cell will tell you whether everything passed. Don't worry not about understanding any of the code here or what it does, that's what this class is for!

**Do not skip cells and do not run the last cell on its own** — it checks that all the earlier cells actually ran, and it will refuse to report success if they did not.

If a cell fails, copy the **full error text** and see the Troubleshooting section of `SETUP.md`.

## 1. Which Python am I running?

The path below should contain your course folder and `.venv`. If it does not, VS Code is using the wrong interpreter — see Step 6 of `SETUP.md`.

In [ ]:
import sys, platform

# Every check below records its result here. The final cell verifies nothing was skipped.
CHECKS = {}

print('Python version :', sys.version.split()[0])
print('Interpreter    :', sys.executable)
print('Operating sys  :', platform.system(), platform.release(), f'({platform.machine()})')

assert sys.version_info[:2] == (3, 13), 'Expected Python 3.13 — see SETUP.md Step 3.'
assert '.venv' in sys.executable, 'Wrong interpreter selected — see SETUP.md Step 6.'

CHECKS['interpreter'] = sys.executable
print('\nInterpreter looks correct.')

## 2. Are all the course libraries installed?

In [ ]:
from importlib.metadata import version, PackageNotFoundError
import importlib

LIBRARIES = [
    ('networkx',   'networkx',     'Week 3  — search'),
    ('ortools',    'ortools',      'Week 4  — constraint programming'),
    ('pulp',       'pulp',         'Week 5  — linear programming'),
    ('pgmpy',      'pgmpy',        'Week 8  — Bayesian networks'),
    ('gymnasium',  'gymnasium',    'Week 9  — reinforcement learning'),
    ('py_trees',   'py-trees',     'Week 10 — behavior trees'),
    ('nashpy',     'nashpy',       'Week 11 — game theory'),
    ('axelrod',    'Axelrod',      'Week 11 — iterated prisoner dilemma'),
    ('mesa',       'mesa',         'Week 11 — agent-based modelling'),
    ('openai',     'openai',       'Week 12 — LLM agents'),
    ('requests',   'requests',     'Week 12 — raw HTTP calls'),
    ('numpy',      'numpy',        'all weeks'),
    ('matplotlib', 'matplotlib',   'all weeks'),
    ('otter',      'otter-grader', 'autograder'),
]

failures = []
for module_name, dist_name, used_in in LIBRARIES:
    try:
        importlib.import_module(module_name)
        try:
            v = version(dist_name)
        except PackageNotFoundError:
            v = '?'
        print(f'  OK    {module_name:<12} {v:<12} {used_in}')
    except Exception as e:
        failures.append(module_name)
        print(f'  FAIL  {module_name:<12} {"":<12} {type(e).__name__}: {e}')

print()
assert not failures, f'Missing: {failures}. Run `uv sync --compile-bytecode` again.'
CHECKS['libraries'] = len(LIBRARIES)
print(f'All {len(LIBRARIES)} course libraries import successfully.')

## 3. Do the solvers actually solve?

Importing a library is not the same as it working. The two cells below run compiled code, which is where machine-specific problems show up.

> ### Windows users: read this if the next cell kills your kernel
> If you see **"The Kernel crashed while executing code in the current cell"** here, your machine is missing a Microsoft system library that OR-Tools needs. This is common and easy to fix. Open PowerShell and run:
>
> ```
> winget install --id Microsoft.VCRedist.2015+.x64
> ```
>
> Then restart VS Code, restart the kernel, and run this notebook again from the top. See Troubleshooting in `SETUP.md` for details.

In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()
x = model.new_int_var(0, 10, 'x')
model.add(x > 3)
model.maximize(x)

solver = cp_model.CpSolver()
status = solver.solve(model)

assert solver.value(x) == 10, 'CP-SAT returned an unexpected answer.'
CHECKS['cp_sat'] = solver.value(x)
print('OR-Tools CP-SAT solver working. x =', solver.value(x))

In [ ]:
import pulp

problem = pulp.LpProblem('check', pulp.LpMaximize)
y = pulp.LpVariable('y', 0, 5)
problem += y
problem.solve(pulp.PULP_CBC_CMD(msg=0))

assert pulp.LpStatus[problem.status] == 'Optimal', 'CBC solver did not run.'
CHECKS['cbc'] = y.value()
print('PuLP + CBC solver working. y =', y.value())

## 4. Do the reinforcement learning environments load?

**Note:** we use `Taxi-v4`. Older tutorials and textbooks refer to `Taxi-v3`, which recent versions of Gymnasium have retired. The problem is identical; only the name changed.

In [ ]:
import gymnasium as gym

for env_id in ['Taxi-v4', 'FrozenLake-v1']:
    env = gym.make(env_id)
    obs, info = env.reset(seed=0)
    print(f'  OK  {env_id:<16} states={env.observation_space.n:<5} actions={env.action_space.n}')
    env.close()

CHECKS['gymnasium'] = ['Taxi-v4', 'FrozenLake-v1']
print('\nGymnasium environments working.')

## 5. Can this notebook draw a plot?

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

G = nx.karate_club_graph()
fig, ax = plt.subplots(figsize=(4, 3))
nx.draw(G, ax=ax, node_size=40, width=0.4)
ax.set_title(f'NetworkX: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
plt.show()

CHECKS['plotting'] = G.number_of_edges()

## 6. Is the autograder working?

In [ ]:
import otter

CHECKS['otter'] = otter.__version__
print('otter-grader version:', otter.__version__)
print('The autograder is installed. You will use it in the weekly labs.')

## 7. Can this notebook see its own folder?

This confirms the notebook can locate itself on disk and read the other files that ship alongside it, like `test.txt`.

In [ ]:
from pathlib import Path

try:
    notebook_dir = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    notebook_dir = Path.cwd()

print('This notebook lives in:', notebook_dir)

test_file = notebook_dir / 'test.txt'
print('\nContents of test.txt:')
print(test_file.read_text())

print('Files in this folder:')
for entry in sorted(notebook_dir.iterdir()):
    print(' ', entry.name)

## Result

This cell verifies that every check above actually ran. It cannot report success on its own.

In [ ]:
import sys, platform
from importlib.metadata import version

REQUIRED = ['interpreter', 'libraries', 'cp_sat', 'cbc', 'gymnasium', 'plotting', 'otter']

try:
    CHECKS
except NameError:
    raise RuntimeError(
        'No checks have run. Start from the top of the notebook and use Run All.'
    ) from None

missing = [name for name in REQUIRED if name not in CHECKS]

if missing:
    raise RuntimeError(
        'ENVIRONMENT CHECK INCOMPLETE.\n'
        f'These checks did not run or did not finish: {missing}\n\n'
        'Scroll up, find the first cell that failed or was skipped, and read its error.\n'
        'If your kernel crashed at the OR-Tools cell and you are on Windows, see the\n'
        'note above that cell. Do not treat this notebook as passed.'
    )

print('=' * 58)
print('  CS-31 ENVIRONMENT CHECK PASSED')
print('=' * 58)
print(f'  OS              : {platform.system()} {platform.release()} ({platform.machine()})')
print(f'  Python          : {sys.version.split()[0]}')
print(f'  Environment     : {sys.executable}')
print(f'  Libraries       : {CHECKS["libraries"]} imported')
print(f'  ortools         : {version("ortools")}')
print(f'  gymnasium       : {version("gymnasium")}')
print(f'  mesa            : {version("mesa")}')
print('=' * 58)
print('\n  Save a copy of this output. If something breaks later,')
print('  it tells us what your machine looked like when it worked.')